# Classification des images spatiales 

Pour commencer, il est utile de présenter notre base de données. On a une base de données d'images spatiales de très hautes résolution (600*600)pixels en RGB (3 canaux), avec 30 différentes classes. 

La problématique de notre bureau d'étude est de trouver le modèle optimale permettant de classer une image correctement dans l'une des 30 classes. Pour répondre à cette problématique nous allons dans un premier temps nous occuper de préparer nos données puis nous allons essayer de trouver le modèle optimale. Ensuite, nous visualiserons nos filtres et analyserons nos résultats. Pour finir, nous allons créer une API.

## I : Préparation des données

Dans un premier temps, nous avons séparé notre base de données en trois partie afin d'obtenir la répartition suivante: 

- **80%** des images sont dans la base d'entraînement
- **10%** des images sont dans la base de validation
- **10%** des images sont dans la base de test

Pour cela nous avons utilisé la fonction "**splitfolder.py**" comme vous pouvez le voir dans la cellule ci-dessous. Nous avons ajustez le ratio afin de correspondre à la répartition voulue et avons laisser la graine aléatoire par défaut "**seed = 1337**". Nous avons compris durant la séance que la graine aléatoire permet de fixer le hasard et permettre de séparer nos images de manière aléatoire.

In [ ]:
import splitfolders 
splitfolders.ratio(r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\AID",output = r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\output",seed = 1337,ratio = (.8,0.1,0.1))

Ensuite, nous avons créer nos modèles de préparations de nos images de tests, validation et d'entrainement. Ces structures permettent en ammont de la classification par notre modèle de préparer nos images, d'effectuer sur ces dernières différentes opérations. C'est à nous de choisir dans le contexte de notre problématique et de nos données à choisir les opérations a effectuer. 

Nous avons décidé dans une premier temps, d'effectuer un "**rescale**" en divisant nos images par 255 afin de normaliser nos images, c'est une étape importante car nous avons des images en RGB et le "**rescale**" permet d'avoir des valeurs dans l'intervalle [0,1]. C'est une étape importante car la mise à l'échelle des données permet au modèle d'apprendre plus rapidement.

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models, optimizers
import numpy as np
import matplotlib.pyplot as plt


train_val = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

train_test = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)


train_train = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

Pour continuer, nous générons automatiquement nos données d'entraînement, de validation et de test pour entrainer notre modèle.
Nos générateurs utilisents des chemins stocker dans la variable "directory" pour accéder aux images. Ensuite, nous avons modifier nos images en utilisant les paramètres suivants : 

- "**target size**" : Les images de notre base de donnée sont de taille 600 * 600 pixels, nous avons remarqué que en pratique entrainer notre modèle avec des données de résolution aussi élevé ralentit l'entrainement. Pour résoudre cette problématique, nous avons dans un premier temps diminué la taile pour atteindre une résolution de 200 * 200 pixels. Finalement nous avons fixé "target size" à  224 * 224 car les couches convolutives du modèle VGG16 que nous utiliserons par la suite ont été entrainé sur des images de cette taille et est donc plus performant pour des images de cette taille.
 
- "**batch_size**" : Ce paramètre fixe le nombre d'images traités en même temps. Nous avons commencé par 64 mais la carte graphique de l'ordinateur a échoué ce qui nous a amené à le fixé à 32, surrement lié au nombre et à la haute résolution de nos images. 
 
- "**color_mode=rgb**" : Nous avons choisi le mode RGB car nos images spatiales sont en RGB.
 
- "**shuffle**" : Le modèle va apprendre avec des images qui ne sont pas dans le meme ordre ce qui permettras d'améliorer la capacité de généralisation.
 
- "**seed**" : En prennant "seed = 42" nous forçons le mélange à rester toujours le même, cela nous permet de s'assurer que si on fait une erreur, ce n'est pas juste parce que le hasard a mal mélangé les images.

In [ ]:
train_generator_train = train_train.flow_from_directory(
    directory=r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\output\train",
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

train_generator_test = train_test.flow_from_directory(
    directory=r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\output\test",
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

train_generator_val = train_val.flow_from_directory(
    directory=r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\output\val",
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

## II : Création et entrainement de notre modèle 

Afin de créer et entraîner notre modèle nous avons suivi la démarche qui suit : 

- Création d'un **premier modèle** qu'on entraine avec notre base de données, nous montrerons qu'il n'est pas optimal.
  
- Utilisation d'un **modèle VG16 pré-entrainer** avec de très grandes base de données, on ne prend que les couches convolutives.

- Utilisation des techniques d'**augmentation des données** pour agrandir notre base de données et améliorer la capacité de généralisation du modèle.

### II.1 : 1er version du modèle et amélioration des paramètres 

Pour notre modèle initial, nous avons construit un réseau de neurones convolutif (CNN) structuré de la manière suivante : 

- Nous avons créer **3 couches convolutives** avec un nombre de filtres ascendant afin d'extraire des carractéristiques de plus en plus complexes sur les images.
  
- Ensuite à la suite de chacune des couche cité précédement nous avons ajouté des couches de normalisation ("**Batch Normalization**") afin de stabiliser l'entraînement.

- Nous ajoutons aussi après chaque couche convolutif, une couche de "**Maxpooling**", afin de réduire la taille des données tout en gardant les informations importantes, permettant donc de garder moins de paramètres.

- Au départ, nous avions ajouté une couche dense intermédiaire avec un Dropout de 50%. Cependant, les résultats n'étaient pas satisfaisants. Nous avons donc garder uniquement la couche de sortie finale de 30 neurones, correspondant à "**NB_CLASSES**" avec une activation **Softmax**. Effectivement, avec la couche dense intermédiaire nous avons systématiquement des **taux d'accuracy** très faible.

Ensuite, pour tester notre modèle nous avons choisi dans un premier temps 5 epochs et 1 de patience (**EarlyStopping**), on a terminé avec 30 Epochs et 3 de patience

Nous avons obtenue avec ces paramètres un accuracy de environ 80%.

In [ ]:
NB_CLASSES = 30
#build the model
model = models.Sequential()
# CONV => RELU => POOL
model.add(layers.Convolution2D(32, (3, 3), padding= 'same', 
           input_shape=(200, 200, 3), activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

# 2 eme couche ajouté
model.add(layers.Convolution2D(70, (3, 3), padding= 'same', 
           input_shape=(200, 200, 3), activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

# 3 eme couvhe
model.add(layers.Convolution2D(100, (3, 3), padding= 'same', 
           input_shape=(200, 200, 3), activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))


# Flatten
model.add(layers.Flatten()) 
# softmax dense classifier
#model.add(layers.Dense(200, activation="relu"))
#model.add(layers.Dropout(0.5))
model.add(layers.Dense(NB_CLASSES, activation="softmax"))
###############
# summary of the model
model.summary()



# compiling the model
model.compile(optimizer='SGD', loss='categorical_crossentropy',
              metrics=['accuracy'])
###############
#training the model
EPOCHS = 20
BATCH_SIZE = 128
VERBOSE = 1

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5)
history = model.fit(train_generator_train, epochs=EPOCHS,
		  verbose=VERBOSE, validation_data=train_generator_val, callbacks=[es])
################
#evaluate the model
test_loss, test_acc = model.evaluate(train_generator_test)
print('Test accuracy:', test_acc)

plt.plot(history.history['val_loss']) 
plt.plot(history.history['loos'])
plt.show()  # affichage fonction loss pour voir si on est pas bloque ds min local


### II.2 : Modèle VG16

Précédement, nous n'arrivions pas à dépasser un taux d'accuracy de environ 80% avec notre modèle. Pour avoir de meilleurs résultats nous avons décidé d'utilisé un **modèle VG16 pré-entrainer** sur de grandes bases de données. Nous allons utilisé dans un premier temps toutes les couches convolutifs de ce modèle et garder nos couches denses. Nous gardons nos couches denses afin d'adapter la classification à nos données. 

In [ ]:
NB_CLASSES = 30
#build the model
# Tunning

base_model = VGG16(include_top=False, weights='imagenet', input_shape=(224, 224, 3))

base_model.summary()

for layer in base_model.layers[:17]:
    layer.trainable = False
# Pour voir quelles sont les couches entraînables
for i, layer in enumerate(base_model.layers):
    print(i, layer.name, layer.trainable)

model = models.Sequential([
    base_model,
    layers.Flatten(), # ou layers.GlobalAveragePooling2D() suivant le cas
    layers.Dense(30, activation='softmax') 
    ])

# compiling the model
model.compile(optimizer='SGD', loss='categorical_crossentropy',
              metrics=['accuracy'])

### II.1: Amélioration par gelage (freezing)

Comme vous pouvez le voir sur le code ci-dessus, nous avons mis en place un **freezing**. Vous pouvez le voir également sur le code ci-dessous que nous avons gelé les 17 premières couches du modèle VGG16 pour conserver ses capacités de reconnaissance tout en gagnant du temps de calcul. Cependant, nous avons laissé les dernières couches libres afin que le modèle VG16 puisse s'entraîner sur nos images et pour qu'il puisse d'adpaté aux spécificités de nos images spatiales et bien classer nos 30 classes. 

In [ ]:
for layer in base_model.layers[:17]:
    layer.trainable = False
# Pour voir quelles sont les couches entraînables
for i, layer in enumerate(base_model.layers):
    print(i, layer.name, layer.trainable)

### II.2: Amélioration par data augmentation et choix 

Nous avons ensuite toujours en vue d'améliorer la capacité de généralisation du modèle et afin d'améliorer la performance du modèle sur nos images, utilisé des techniques d'**augmentation de données**. 

In [ ]:
train_val = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,

)

train_test = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)


train_train = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True,
    rotation_range=5,

)

Pour augmenter les données nous avons ajouté dans notre générateur d'entraînement, qui permettent de modifier en ammont du modèle nos images ajouté de nouvelles opérations sur images. Nous avons uniquement mis en places les opérations suivante sugéré dans les support séance : 

- **Décalage** : Nous avons appliqué de légers décalages horizontaux/verticaux à l'aide des variable "**width_shift et height_shift**" afin d'apprendre au modèle qu'une image reste la même même si elle a subit un léger décalage. En ce sens cela permet d'améliorer aussi la robustesse du modèle. 

- **Rotation** : Nous avons aussi mis en place une rotation de 5° ("**horizontal_flip**"). Cela permet aussi au modèle à reconnaître une image si elle a subit une légère rotation.

Cependant nous avons décidé de ne pas garder les opérations suivantes sugéré dans le support de séance : 

- **Luminance** : Nous avons enlevé la luminance car nos images sont de très haute résolution et rajouter de la luminance ne rajoutera rien de pertinant pour l'apprentissage et nous avons eu peur que cela ajoute des erreurs de classifications car on suppose que dans les couches convolutifs il y a des filtres sur les couleurs. Et la variable "**Brightness**" pourrait perturber ces filtresenmodifiant les luminosité sur les images.
  
- **Zoom** : Nous avons également enlevé le **Zoom** car on a supposé qu'il peut introduire un mélange entre les classes. Effectivement, nous avons des classes qui se ressemblent comme quartier résidentiel et ecoles et si nous zoomons sur un bâtiment en particulier cela pourrait entrîner des erreurs de classifications suplémentaires.

Nous pourrons mieux mettre en évidence ces hyppothèses formulé lorsque nous visualiserons nos filtres par la suite.

### II.3: Amélioration de la performance et analyse des résultats intermédiare

Nous avons vu à la suite du **Data Augmentation** et du **Freezing** une nette amélioration du taux d'accuracy qui avoisine les 90%. Pour encore améliorer les performance du modèle, nous avons enregistré les meilleurs paramètres avant même la fin des EPOCHS, comme vous pouvez le voir sur le code ci-dessous.

In [ ]:
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=3,restore_best_weights=True)

model_checkpoint = ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)

callbacks = [es, model_checkpoint]



history = model.fit(train_generator_train, epochs=EPOCHS,
		  verbose=VERBOSE, validation_data=train_generator_val, callbacks=callbacks)
################
#evaluate the model
test_loss, test_acc = model.evaluate(train_generator_test)
print('Test accuracy:', test_acc)

plt.plot(history.history['val_loss']) 
plt.plot(history.history['loss'])
plt.show()  # affichage fonction loss pour voir si on est pas bloque ds min local

Pour sauvegarder le meilleur modèle, nous avons utilisé la fonction **Model_checkpoint**. En activant l'option "**save_best_only=True**", on compare à chaque fois le résultat de performance de l'epoque actuelle avec les scores historiquele. Si il y a une amélioration alors le fichier best_model.h5 est mis à jour. Cela permet de conserver le meilleur modèles avec les meilleurs paramètres.

Nous avons pour finir enregistré notre modèle afin de pouvoir l'exploiter quand on le veut. Pour cela, nous avons fait la sauvegarde ci-dessous : 

In [ ]:
#save du model
model_json = model.to_json()
with open ('model.json','w') as json_file:
    json_file.write(model_json)
    model.save_weights('model.h5')

## III : Visualisation des filtres et des caractéristiques extraites par le réseau

Pour continuer, maintenant que nous avons notre modèle enregistré nous l'avons charger dans un nouveau script afin de visualisé le comportement de certains filtres des couches convolutives du modéle VG16. Plus particulièrement, nous avons extrait et normalisé les poids de la première couche convolutive du modèle afin de visualiser les filtres.


Afin de visualiser les différents filtres, nous avons mis en oeuvre le code suivant : 

In [ ]:
from tensorflow.keras.models import model_from_json
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import cv2
from sklearn.metrics import confusion_matrix, classification_report
from keras.models import Model
import tensorflow as tf
import seaborn as sns
from tensorflow.keras.applications.vgg19 import VGG19

In [ ]:
train_test = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

train_generator_test = train_test.flow_from_directory(
    directory=r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\output\test",
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="categorical",
    shuffle=False,
    seed=42
)

model_architecture = 'model.json'
model_weights = 'best_model.h5'
model = model_from_json(open(model_architecture).read())
model.load_weights(model_weights)
model.summary()

base_model = model.layers[0]
base_model.summary()

# Récupérer les filtres et les poids de la 1ère couche
filters, biases, *is_anything_else_being_returned = model.layers[0].get_weights()
# normaliser les filtres sur [0 , 1]
f_min, f_max = filters.min(), filters.max()
filters = (filters - f_min) / (f_max - f_min)

# Prendre le deuxième filtre de cette couche (qui contient 3 sous-filtres pour les 3 canaux R,
# V et et B) et visualiser ces 3 sous-filtres
plt.figure()
f=filters[:,:,:,1]
plt.subplot(1,3,1)
plt.imshow(f[:, :, 0], cmap='gray')
plt.subplot(1,3,2)
plt.imshow(f[:, :, 1], cmap='gray')
plt.subplot(1,3,3)
plt.imshow(f[:, :, 2], cmap='gray')
plt.close()


Nous avons ensuite remarqué que le résultat normalisé précédent n'était pas suffisant pour comprendre les fonctionnalités des filtres. Pour mieux comprendre nos filtres, nous avons créé un modèle intermédiaire afin de visualiser des **cartes de caractéristiques** en faisant passer des vrais images dans le modèle afin de mieux visualiser le résultat des filtres. 

In [ ]:
fichier = r"C:\Users\ML\Desktop\G1_SIA2_images_spatiales\AID\Stadium\stadium_83.jpg"
image = Image.open(fichier)
# #im_array = np.array(image)
# #im_fl = np.float64(im_array)
# #new_shape = (224,224,3)
# im_res=np.resize(im_fl, new_shape)/255.0
# im_fin = np.reshape(im_res, (1,224,224,3))
image = np.asarray(image)
image=image.astype('float32')
image=cv2.resize(image,(224,224))
image /= 255
image = image.reshape([-1,224,224,3])


# Définir un modèle intermédiaire contenant les 2 premières couches du modèle initial

inter_model = Model(inputs=base_model.inputs, outputs=base_model.layers[1].output)
inter_model.summary()
feature_maps = inter_model.predict(image)
plt.figure()
plt.imshow(feature_maps[0,:,:,0])

Nous avons fais le test sur plusieurs images différentes et avons remarqué essentiellement que :

- **La couche 4** ne fait rien, il n'y a aucune activation.
  
- **La première couche** extrait les contours, on a bien visualiser ça sur les images des aéroport où nous avons bien visualisé les cotnours des avions.

- **La seconde couche** traite des informations plus globales comme la couleur.


A noter que les conclusions effectué plus haut sont seulement celles que nous avons formulé après observation de quelques images. Pour affiner notre compréhension des filtres, nous aurions dû tester plus d'images de catégorie plus varié. 


## IV: Analyse des résultats

Pour continuer, nous avons évalué les performances finales du modèle en calculant sa précision, sa matrice de confusion. Nous avons grâce à la matrice de confusion pu identifier les classes les mieux classer comme celle de "**Viaduc**" ou encore les moins bien classé comme celle de "**School**".

In [ ]:
# Rapport classification
np.max(history.history['val_accuracy'])
Y_pred = model.predict(train_generator_test)
yy_pred = np.argmax(Y_pred,1)
yy_test = np.argmax(train_generator_test,1)
mat = confusion_matrix(yy_test,yy_pred)
sns.heatmap(mat.T,square=True,annot=True,cbar=False,cmap=plt.cm.Blues,fmt='.0f')
plt.xlabel('valeur prédite')
plt.ylabel('valeurs réelles')
plt.show()

model_architecture = 'model.json'
model_weights = 'best_model.h5'
model = model_from_json(open(model_architecture).read())
model.load_weights(model_weights)

## V: Création d'une API

Enfin, pour la partie finale, nous devions faire une API, ce qui signifie "application programming interface" ou en français "interface de programmation d'application".
Pour cela, nous avons dû utiliser un exemple donné en TP et le configurer en fonction de nos classes dans notre fichier **streamlit_sat.py**. 

In [ ]:
import streamlit as st
import cv2
from PIL import Image, ImageOps
import numpy as np
from tensorflow.keras.models import model_from_json

nom_classe=["Airport","BareLand","BaseballField","Beach","Bridge","Center","Church","Commercial","DenseResidential","Desert"
            , "Farmland", "Forest", "Industrial", "Meadow", "MediumResidential", "Mountain", "Park", "Parking", "Playground", "Pond", 
            "Port", "RailwayStation", "Resort", "River", "School","SparseResidential","Square", "Stadium", 
            "StorageTanks", "Viaduct"]

@st.cache_data
def load_model():
    model_architecture = 'model.json'
    model_weights = 'best_model.h5'
    model = 	model_from_json(open(model_architecture).read())
    model.load_weights(model_weights)  
    return model

with st.spinner('Model is being loaded..'):
  model=load_model()
 
st.write("""
         # Classification des images sat.         """
         )
 
file = st.file_uploader("Upload the image to be classified", type=["jpg", "png", "jpeg"])
st.set_option('deprecation.showfileUploaderEncoding', False)
 
def upload_predict(image, model):  
        image = np.asarray(image)
        image=image.astype('float32')
        image=cv2.resize(image,(224,224))
        image /= 255
        image = image.reshape([-1,224,224,3])
        prediction = model.predict(image)
        pred_class=np.argmax(prediction, axis=1)       
        return pred_class, prediction
    
if file is None:
    st.text("Please upload an image file")
else:
    image = Image.open(file)
    st.image(image, use_column_width=True)
    pred_class, prediction = upload_predict(image, model)
    st.write("The image is classified as",nom_classe[pred_class[0]])
    st.write("The probability is", prediction[0][pred_class[0]])


Pour que tout fonctionne, nous devions modifier quelques lignes :
- Changer les noms des classes, en y rajoutant nos 30 classes.
- Modifier les modèles en entrée, ici il fallaît juste les remplacer par les nôtres (model.json et best_model.h5)
- Modifier les tailles des images pour respecter les bases du VGG16, à savoir 224 x 224.

Enfin, pour lancer notre fichier, il fallait utiliser la commande **streamlit run streamlit_sat.py** pour que notre API se lance.